# LightGCN trên BPATMP (REES46)

Notebook chạy **LightGCN** (https://github.com/gusye1234/LightGCN-PyTorch) trên dataset BPATMP với:

### Training data (graph + BPR sampling)
| Behavior | Số edges |
|---|---|
| `view` | subsample ngẫu nhiên **3,000,000** edges |
| `cart` | **full** (toàn bộ) |
| `purchase` | **full** (toàn bộ) |

Ba tập gộp + dedup → một interaction matrix duy nhất cho LightGCN.

### Evaluation
- Ground-truth = **purchase only** (`val_ground_truth`, `test_ground_truth`)
- Mask: `train_mask_purchase_only` (loại purchase train items khi scoring)
- **HR@k** & **NDCG@k** tại k = 1, 5, 10, 20, 50 — full-rank
- Lưu `best.pt` + `last.pt` | Early stopping theo `NDCG@20`

## 1. Cài đặt môi trường + clone LightGCN-PyTorch

In [1]:
!pip install -q pyyaml loguru
import os, sys

LIGHTGCN_DIR = '/content/LightGCN-PyTorch'
if not os.path.isdir(LIGHTGCN_DIR):
    !git clone --depth 1 https://github.com/gusye1234/LightGCN-PyTorch.git {LIGHTGCN_DIR}

# Không import world/dataloader gốc — notebook dùng LightGCN self-contained.
print('LightGCN repo files:', os.listdir(os.path.join(LIGHTGCN_DIR, 'code')))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
Cloning into '/content/LightGCN-PyTorch'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 45 (delta 1), reused 31 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 40.91 MiB | 22.96 MiB/s, done.
Resolving deltas: 100% (1/1), done.
LightGCN repo files: ['dataloader.py', 'model.py', 'register.py', 'Procedure.py', 'main.py', 'world.py', 'parse.py', 'utils.py', 'sources']


## 2. Tải dataset BPATMP từ Hugging Face

In [2]:
import os
DATA_DIR = '/content/data'

if not os.path.isfile(os.path.join(DATA_DIR, 'purchase_train_src.npy')):
    !pip install -q huggingface_hub hf_transfer
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='nguyenmaiductrong/rees46-bpatmp-temporal',
        repo_type='dataset',
        local_dir=DATA_DIR,
    )
print('data files:', sorted(os.listdir(DATA_DIR))[:25])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 33.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 167 files:   0%|          | 0/167 [00:00<?, ?it/s]

data files: ['.cache', '.gitattributes', 'artifacts_manifest.json', 'baseline_contract.md', 'candidate_item_idx.npy', 'cart_train_dst.npy', 'cart_train_src.npy', 'cart_train_ts.npy', 'data_card.md', 'graph', 'node_counts.json', 'node_mappings', 'purchase_train_dst.npy', 'purchase_train_src.npy', 'purchase_train_ts.npy', 'split_manifest.json', 'test_ground_truth.pkl', 'test_product_idx.npy', 'test_timestamp.npy', 'test_user_idx.npy', 'train_mask.pkl', 'train_mask_purchase_only.pkl', 'train_mask_seen_all.pkl', 'val_ground_truth.pkl', 'val_product_idx.npy']


## 3. Cấu hình Hyperparameters

In [30]:
import yaml, io, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

YAML_TEXT = """
data:
  data_dir: /content/data/
  node_counts:
    product: 29892
    user: 203063
model:
  embed_dim: 64
  n_layers: 3
  keep_prob: 0.6
  A_split: False
training:
  batch_size: 2048
  eval_batch_size: 512
  epochs: 40
  lr: 0.001
  l2_lambda: 0.0001
  seed: 42
  eval_every: 1
  patience: 5
  device: cuda
  save_dir: /content/checkpoints-lightgcn
  max_view_edges: 3000000
  max_cart_edges: 0
  max_purchase_edges: 0
evaluation:
  primary_metric: NDCG@20
  ks:
  - 1
  - 5
  - 10
  - 20
  - 50
"""

yaml_path = os.path.join(DATA_DIR, 'training.yaml')
if os.path.isfile(yaml_path):
    with open(yaml_path) as f:
        CFG = yaml.safe_load(f)
    CFG['model'].setdefault('n_layers', 3)
    CFG['model'].setdefault('keep_prob', 0.6)
    CFG['model'].setdefault('A_split', False)
    T = CFG.setdefault('training', {})
    T.setdefault('eval_batch_size', 512)
    T.setdefault('max_view_edges',     3_000_000)
    T.setdefault('max_cart_edges',     0)
    T.setdefault('max_purchase_edges', 0)
    CFG['evaluation'] = CFG.get('evaluation', {})
    CFG['evaluation']['ks'] = [1, 5, 10, 20, 50]
    CFG['evaluation'].setdefault('primary_metric', 'NDCG@20')
else:
    CFG = yaml.safe_load(YAML_TEXT)

# Giới hạn Colab
if CFG['model'].get('embed_dim', 64) > 64:
    print('[adjust] embed_dim -> 64'); CFG['model']['embed_dim'] = 64
if CFG['model'].get('n_layers', 3) > 4:
    print('[adjust] n_layers -> 3');  CFG['model']['n_layers'] = 3
if CFG['training'].get('batch_size', 2048) > 4096:
    CFG['training']['batch_size'] = 4096

os.makedirs(CFG['training']['save_dir'], exist_ok=True)
print(yaml.safe_dump(CFG, sort_keys=False))

data:
  data_dir: /content/data/
  node_counts:
    product: 29892
    user: 203063
model:
  embed_dim: 64
  n_layers: 3
  keep_prob: 0.6
  A_split: false
training:
  batch_size: 2048
  eval_batch_size: 512
  epochs: 40
  lr: 0.001
  l2_lambda: 0.0001
  seed: 42
  eval_every: 1
  patience: 5
  device: cuda
  save_dir: /content/checkpoints-lightgcn
  max_view_edges: 3000000
  max_cart_edges: 0
  max_purchase_edges: 0
evaluation:
  primary_metric: NDCG@20
  ks:
  - 1
  - 5
  - 10
  - 20
  - 50



## 4. Load BPATMP data — view (3M) + cart (full) + purchase (full)

**Chiến lược:**
1. Load từng behavior, subsample view nếu vượt cap
2. Ghép tất cả `(user, item)` pairs → **dedup** bằng `np.unique`
3. Interaction gộp này làm input cho LightGCN graph + BPR sampling
4. Ground-truth eval vẫn **purchase-only**

In [24]:
import numpy as np, pickle, os

DATA    = CFG['data']['data_dir']
N_USERS = CFG['data']['node_counts']['user']    # 203063
N_ITEMS = CFG['data']['node_counts']['product'] # 29892
RNG     = np.random.default_rng(CFG['training']['seed'])


def load_behavior(name, cap=0):
    src = np.load(os.path.join(DATA, f'{name}_train_src.npy')).astype(np.int64)
    dst = np.load(os.path.join(DATA, f'{name}_train_dst.npy')).astype(np.int64)
    raw = len(src)
    if cap and raw > cap:
        sel = RNG.choice(raw, size=cap, replace=False)
        src, dst = src[sel], dst[sel]
        print(f'  {name:10s}: {raw:>10,} edges  -> subsampled to {cap:,}')
    else:
        print(f'  {name:10s}: {raw:>10,} edges  (full)')
    return src, dst


print('Loading behavior edges...')
view_src, view_dst = load_behavior('view',     cap=CFG['training']['max_view_edges'])
cart_src, cart_dst = load_behavior('cart',     cap=CFG['training']['max_cart_edges'])
pur_src,  pur_dst  = load_behavior('purchase', cap=CFG['training']['max_purchase_edges'])

# ── Gộp + dedup ─────────────────────────────────────────────
all_src = np.concatenate([view_src, cart_src, pur_src])
all_dst = np.concatenate([view_dst, cart_dst, pur_dst])
print(f'\nTong truoc dedup : {len(all_src):,}')

pairs      = np.stack([all_src, all_dst], axis=1)  # (N, 2)
pairs_uni  = np.unique(pairs, axis=0)              # sort + dedup
train_users = pairs_uni[:, 0]
train_items = pairs_uni[:, 1]
print(f'Sau dedup        : {len(train_users):,} unique (user, item) pairs')
print(f'  user range     : [{train_users.min()}, {train_users.max()}]')
print(f'  item range     : [{train_items.min()}, {train_items.max()}]')

# Thống kê overlap
def to_set(s, d): return set(zip(s.tolist(), d.tolist()))
view_set = to_set(view_src, view_dst)
cart_set = to_set(cart_src, cart_dst)
pur_set  = to_set(pur_src,  pur_dst)
print(f'\nUnique pairs per behavior:')
print(f'  view={len(view_set):,}  cart={len(cart_set):,}  purchase={len(pur_set):,}')
print(f'Overlap: view+cart={len(view_set & cart_set):,}  '
      f'view+pur={len(view_set & pur_set):,}  '
      f'cart+pur={len(cart_set & pur_set):,}')

# ── Eval data ───────────────────────────────────────────────
with open(os.path.join(DATA, 'train_mask_purchase_only.pkl'), 'rb') as f:
    TRAIN_MASK = pickle.load(f)
with open(os.path.join(DATA, 'val_ground_truth.pkl'), 'rb') as f:
    VAL_GT = pickle.load(f)
with open(os.path.join(DATA, 'test_ground_truth.pkl'), 'rb') as f:
    TEST_GT = pickle.load(f)

print(f'\nEval: val={len(VAL_GT):,} users  test={len(TEST_GT):,} users  '
      f'train_mask={len(TRAIN_MASK):,} users')

Loading behavior edges...
  view      : 25,956,733 edges  -> subsampled to 3,000,000
  cart      :  3,868,050 edges  (full)
  purchase  :  2,436,760 edges  (full)

Tong truoc dedup : 9,304,810
Sau dedup        : 3,306,742 unique (user, item) pairs
  user range     : [0, 203062]
  item range     : [0, 29891]

Unique pairs per behavior:
  view=2,308,829  cart=1,578,188  purchase=1,308,730
Overlap: view+cart=669,511  view+pur=571,449  cart+pur=1,168,899

Eval: val=47,996 users  test=23,536 users  train_mask=58,579 users


## 5. Dataset Adapter cho LightGCN

Interface: `n_users`, `m_items`, `trainDataSize`, `allPos`, `getSparseGraph()`.  
Index **0-based**. Graph input = combined interactions (view 3M + cart full + purchase full, dedup).

In [25]:
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import torch, numpy as np


class BPATMPDatasetLightGCN:
    """
    Adapter cho LightGCN.
    Graph + BPR sampling dung combined interactions (view+cart+purchase, dedup).
    Index 0-based (LightGCN convention).
    """
    def __init__(self, n_users, n_items, train_users, train_items):
        self._n_users = n_users
        self._m_items = n_items
        self.trainUser = train_users
        self.trainItem = train_items
        self._trainDataSize = len(train_users)
        self.Graph = None

        self.UserItemNet = csr_matrix(
            (np.ones(self._trainDataSize, dtype=np.float32),
             (train_users, train_items)),
            shape=(n_users, n_items)
        )

        print('Building allPos lookup...')
        self._allPos = self.getUserPosItems(list(range(n_users)))
        sparsity = self._trainDataSize / (n_users * n_items)
        print(f'Dataset ready: {n_users:,} users | {n_items:,} items | '
              f'{self._trainDataSize:,} interactions | sparsity={sparsity:.6f}')

    @property
    def n_users(self):       return self._n_users
    @property
    def m_items(self):       return self._m_items
    @property
    def trainDataSize(self): return self._trainDataSize
    @property
    def allPos(self):        return self._allPos

    def getUserPosItems(self, users):
        return [self.UserItemNet[u].indices.copy() for u in users]

    def getSparseGraph(self):
        """
        Normalized adjacency: A_hat = D^{-1/2} A D^{-1/2}
        A = [[0, R], [R^T, 0]]  (bipartite user-item)
        """
        if self.Graph is not None:
            return self.Graph

        print('Building normalized adjacency...')
        R   = self.UserItemNet
        adj = sp.bmat([[None, R], [R.T, None]], format='csr', dtype=np.float32)

        rowsum   = np.array(adj.sum(axis=1)).flatten()
        d_inv_sq = np.where(rowsum > 0, rowsum ** -0.5, 0.0).astype(np.float32)
        D_inv_sq = sp.diags(d_inv_sq)
        norm_adj = (D_inv_sq @ adj @ D_inv_sq).tocoo().astype(np.float32)

        idx = torch.from_numpy(np.vstack([norm_adj.row, norm_adj.col]).astype(np.int64))
        val = torch.from_numpy(norm_adj.data)
        n   = self._n_users + self._m_items
        self.Graph = torch.sparse_coo_tensor(idx, val, (n, n)).coalesce()
        print(f'Graph built: ({n}, {n}), nnz={norm_adj.nnz:,}')
        return self.Graph


dataset = BPATMPDatasetLightGCN(N_USERS, N_ITEMS, train_users, train_items)
_ = dataset.getSparseGraph()  # pre-build

Building allPos lookup...
Dataset ready: 203,063 users | 29,892 items | 3,306,742 interactions | sparsity=0.000545
Building normalized adjacency...
Graph built: (232955, 232955), nnz=6,613,484


## 6. LightGCN Model (self-contained, no world.py)

In [26]:
import torch, torch.nn as nn, numpy as np


class LightGCN(nn.Module):
    """
    LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation.
    He et al., SIGIR 2020.  Self-contained, khong phu thuoc world.py.
    """
    def __init__(self, config, dataset):
        super().__init__()
        self.num_users  = dataset.n_users
        self.num_items  = dataset.m_items
        self.latent_dim = config['latent_dim_rec']
        self.n_layers   = config['lightGCN_n_layers']
        self.keep_prob  = config.get('keep_prob', 0.6)
        self.A_split    = config.get('A_split', False)
        self.use_drop   = config.get('dropout', False)

        self.embedding_user = nn.Embedding(self.num_users, self.latent_dim)
        self.embedding_item = nn.Embedding(self.num_items, self.latent_dim)
        nn.init.normal_(self.embedding_user.weight, std=0.1)
        nn.init.normal_(self.embedding_item.weight, std=0.1)

        self.f     = nn.Sigmoid()
        self.Graph = dataset.getSparseGraph()  # CPU sparse tensor
        print(f'LightGCN | layers={self.n_layers} | dim={self.latent_dim} | dropout={self.use_drop}')

    def _dropout_x(self, x, keep_prob):
        idx  = x.indices().t()
        vals = x.values()
        mask = (torch.rand(len(vals)) + keep_prob).int().bool()
        return torch.sparse_coo_tensor(idx[mask].t(), vals[mask] / keep_prob, x.size()).coalesce()

    def computer(self):
        all_emb = torch.cat([self.embedding_user.weight, self.embedding_item.weight])
        embs    = [all_emb]

        g = self._dropout_x(self.Graph, self.keep_prob) if (self.use_drop and self.training) \
            else self.Graph
        g = g.to(all_emb.device)

        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(g, all_emb)
            embs.append(all_emb)

        light_out = torch.mean(torch.stack(embs, dim=1), dim=1)
        return torch.split(light_out, [self.num_users, self.num_items])

    def getUsersRating(self, users):
        u, i = self.computer()
        return self.f(u[users.long()] @ i.t())

    def getEmbedding(self, users, pos_items, neg_items):
        u, i = self.computer()
        return (u[users], i[pos_items], i[neg_items],
                self.embedding_user(users),
                self.embedding_item(pos_items),
                self.embedding_item(neg_items))

    def bpr_loss(self, users, pos, neg):
        u_e, p_e, n_e, u0, p0, n0 = self.getEmbedding(users.long(), pos.long(), neg.long())
        reg = (1/2) * (u0.norm(2).pow(2) + p0.norm(2).pow(2) + n0.norm(2).pow(2)) / len(users)
        loss = torch.mean(nn.functional.softplus((u_e * n_e).sum(1) - (u_e * p_e).sum(1)))
        return loss, reg

    def forward(self, users, items):
        u, i = self.computer()
        return (u[users] * i[items]).sum(dim=1)


import random
SEED = CFG['training']['seed']
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = CFG['training']['device'] if torch.cuda.is_available() else 'cpu'
lgn_config = {
    'latent_dim_rec':    CFG['model']['embed_dim'],
    'lightGCN_n_layers': CFG['model']['n_layers'],
    'keep_prob':         CFG['model'].get('keep_prob', 0.6),
    'A_split':           CFG['model'].get('A_split', False),
    'dropout':           False,  # doi thanh True de bat dropout
}
model = LightGCN(lgn_config, dataset).to(device)
print(f'Model on [{device}] | params = {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

LightGCN | layers=3 | dim=64 | dropout=False
Model on [cuda] | params = 14.91M


## 7. BPR Sampler

Sampling từ **combined interactions** (view 3M + cart full + purchase full, dedup).  
Negative item: uniform random, loại tất cả items user đã tương tác (any behavior).

In [27]:
import numpy as np
from torch.utils.data import Dataset, DataLoader


class BPRDataset(Dataset):
    """
    (user, pos_item, neg_item) — 0-based.
    pos lay tu combined interactions; neg sampling loai tat ca positive (any behavior).
    """
    def __init__(self, dataset):
        self.n_items   = dataset.m_items
        self.trainUser = dataset.trainUser
        self.trainItem = dataset.trainItem
        print('Building pos_set for neg sampling...')
        self.pos_set = [set(arr.tolist()) for arr in dataset.allPos]
        print('Done.')

    def __len__(self):
        return len(self.trainUser)

    def __getitem__(self, idx):
        user = int(self.trainUser[idx])
        pos  = int(self.trainItem[idx])
        neg  = np.random.randint(0, self.n_items)
        ps   = self.pos_set[user]
        for _ in range(10):
            if neg not in ps:
                break
            neg = np.random.randint(0, self.n_items)
        return np.array([user, pos, neg], dtype=np.int64)


train_ds = BPRDataset(dataset)
train_loader = DataLoader(
    train_ds,
    batch_size=CFG['training']['batch_size'],
    shuffle=True, num_workers=2,
    pin_memory=(device == 'cuda'), drop_last=False,
)
print(f'train_loader: {len(train_loader):,} batches/epoch | '
      f'total interactions (combined): {len(train_ds):,}')

Building pos_set for neg sampling...
Done.
train_loader: 1,615 batches/epoch | total interactions (combined): 3,306,742


## 8. Evaluator — HR@k & NDCG@k (full-rank, mask exclusion)

- Score toàn bộ N_ITEMS items cho mỗi user  
- Mask **purchase train items** (`train_mask_purchase_only`)  
- Ground-truth = **purchase-only**  
- k = **1, 5, 10, 20, 50**

In [28]:
import torch, numpy as np


@torch.no_grad()
def evaluate(model, gt_dict, exclude_dict,
             ks=(1, 5, 10, 20, 50), user_batch=512, item_tile=16384):
    model.eval()
    max_k = max(ks)
    dev   = next(model.parameters()).device
    dtype = torch.float16 if dev.type == 'cuda' else torch.float32

    # Pre-compute embeddings
    all_u, all_i = model.computer()
    all_u = all_u.to(dtype)
    all_i = all_i.to(dtype)

    eval_uids = np.array(sorted(gt_dict.keys()), dtype=np.int64)
    n_eval    = len(eval_uids)

    # Ground-truth padding
    gt_lists = [np.asarray(gt_dict[u], dtype=np.int64).reshape(-1) for u in eval_uids]
    max_pos  = max(len(g) for g in gt_lists)
    gt_pad   = torch.full((n_eval, max_pos), -1, dtype=torch.long, device=dev)
    gt_cnt   = torch.zeros(n_eval, dtype=torch.long, device=dev)
    for r, g in enumerate(gt_lists):
        gt_cnt[r] = len(g)
        gt_pad[r, :len(g)] = torch.as_tensor(g, device=dev)

    # Exclusion sparse index
    uid2pos = {int(u): i for i, u in enumerate(eval_uids.tolist())}
    er, ec = [], []
    for u, items in exclude_dict.items():
        p = uid2pos.get(int(u))
        if p is None or len(items) == 0: continue
        arr = np.asarray(items, dtype=np.int64).reshape(-1)
        er.extend([p] * len(arr)); ec.extend(arr.tolist())
    er = torch.as_tensor(er, dtype=torch.long, device=dev)
    ec = torch.as_tensor(ec, dtype=torch.long, device=dev)

    ndcg_w = 1.0 / torch.log2(
        torch.arange(1, max_k + 1, device=dev, dtype=torch.float32) + 1.0)
    sums = {f'{m}@{k}': 0.0 for m in ('HR', 'NDCG') for k in ks}
    n_items = all_i.size(0)

    for us in range(0, n_eval, user_batch):
        ue    = min(us + user_batch, n_eval)
        B     = ue - us
        uids  = torch.as_tensor(eval_uids[us:ue], dtype=torch.long, device=dev)
        u_emb = all_u[uids]

        top_v = torch.full((B, max_k), float('-inf'), device=dev, dtype=dtype)
        top_i = torch.full((B, max_k), -1,            device=dev, dtype=torch.long)

        inb = (er >= us) & (er < ue)
        sr, sc = er[inb] - us, ec[inb]

        for ts in range(0, n_items, item_tile):
            te     = min(ts + item_tile, n_items)
            scores = u_emb @ all_i[ts:te].t()
            tmask  = (sc >= ts) & (sc < te)
            if tmask.any():
                scores[sr[tmask], sc[tmask] - ts] = float('-inf')
            k_tile = min(max_k, scores.size(1))
            tv, ti = scores.topk(k_tile, dim=-1)
            ti     = ti + ts
            mv = torch.cat([top_v, tv], dim=-1)
            mi = torch.cat([top_i, ti], dim=-1)
            sel   = mv.topk(max_k, dim=-1).indices
            top_v = mv.gather(1, sel)
            top_i = mi.gather(1, sel)

        gtb  = gt_pad[us:ue]
        cnb  = gt_cnt[us:ue]
        hits = (top_i.unsqueeze(-1) == gtb.unsqueeze(1)).any(dim=-1)

        for k in ks:
            sums[f'HR@{k}']   += hits[:, :k].any(dim=-1).float().sum().item()
            dcg  = (hits[:, :k].float() * ndcg_w[:k]).sum(dim=-1)
            idl  = torch.minimum(cnb, torch.full_like(cnb, k))
            idcg = torch.zeros_like(dcg)
            for v in idl.unique():
                mm = (idl == v)
                if int(v) > 0: idcg[mm] = ndcg_w[:int(v)].sum()
            sums[f'NDCG@{k}'] += (dcg / idcg.clamp_min(1e-12)).sum().item()

    return {k: v / n_eval for k, v in sums.items()}


print('evaluate() defined  |  HR + NDCG @ k = 1, 5, 10, 20, 50')

evaluate() defined  |  HR + NDCG @ k = 1, 5, 10, 20, 50


## 9. Training Loop — Save Best Epoch & Early Stopping

In [31]:
import time, gc, os, torch

ks         = CFG['evaluation']['ks']
primary    = CFG['evaluation']['primary_metric']
save_dir   = CFG['training']['save_dir']
patience   = CFG['training']['patience']
l2_lambda  = CFG['training']['l2_lambda']
eval_every = CFG['training']['eval_every']
n_epochs   = CFG['training']['epochs']

best_ckpt_path = os.path.join(save_dir, 'best.pt')
last_ckpt_path = os.path.join(save_dir, 'last.pt')

optim = torch.optim.Adam(model.parameters(), lr=CFG['training']['lr'])

best_metric = -1.0
best_epoch  = -1
stale       = 0


def free_vram():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def print_metrics(phase, m, ks):
    r1 = '  '.join(f'HR@{k}={m[f"HR@{k}"]:.4f}'   for k in ks)
    r2 = '  '.join(f'ND@{k}={m[f"NDCG@{k}"]:.4f}' for k in ks)
    print(f'  [{phase}] {r1}')
    print(f'          {r2}')


print(f'Training LightGCN')
print(f'  graph = view(3M) + cart(full) + purchase(full) -> {len(train_ds):,} combined interactions')
print(f'  epochs={n_epochs}  patience={patience}  primary={primary}')
print('=' * 70)

for epoch in range(1, n_epochs + 1):
    model.train()
    t0 = time.time()
    ep_loss, nb = 0.0, 0

    for batch in train_loader:
        batch = batch.to(device)
        users, pos, neg = batch[:, 0], batch[:, 1], batch[:, 2]
        bpr_l, reg_l = model.bpr_loss(users, pos, neg)
        loss = bpr_l + l2_lambda * reg_l
        optim.zero_grad(); loss.backward(); optim.step()
        ep_loss += float(loss.item()); nb += 1
        del batch, loss, bpr_l, reg_l

    free_vram()
    avg_loss = ep_loss / max(nb, 1)
    dt = time.time() - t0
    print(f'Epoch {epoch:03d}/{n_epochs} | loss={avg_loss:.4f} | {dt:.1f}s')

    # Luu last checkpoint
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(), 'loss': avg_loss,
    }, last_ckpt_path)

    if epoch % eval_every == 0:
        m = evaluate(model, VAL_GT, TRAIN_MASK, ks=ks,
                     user_batch=CFG['training']['eval_batch_size'])
        free_vram()
        print_metrics('val', m, ks)
        cur = m[primary]

        if cur > best_metric:
            best_metric = cur; best_epoch = epoch; stale = 0
            torch.save({
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optim.state_dict(),
                primary: best_metric, 'all_val_metrics': m,
            }, best_ckpt_path)
            print(f'  NEW BEST {primary}={cur:.4f} @ epoch {epoch}  -> {best_ckpt_path}')
        else:
            stale += 1
            print(f'  No improvement ({stale}/{patience}) | best={best_metric:.4f} @ epoch {best_epoch}')
            if stale >= patience:
                print(f'\nEarly stopping at epoch {epoch}.')
                print(f'Best epoch={best_epoch} | {primary}={best_metric:.4f}')
                break

print(f'\n{"="*70}')
print(f'Done. Best {primary} = {best_metric:.4f} @ epoch {best_epoch}')

Training LightGCN
  graph = view(3M) + cart(full) + purchase(full) -> 3,306,742 combined interactions
  epochs=40  patience=5  primary=NDCG@20
Epoch 001/40 | loss=0.8443 | 525.2s
  [val] HR@1=0.0370  HR@5=0.1270  HR@10=0.1865  HR@20=0.2523  HR@50=0.3443
          ND@1=0.0370  ND@5=0.0524  ND@10=0.0652  ND@20=0.0784  ND@50=0.0952
  NEW BEST NDCG@20=0.0784 @ epoch 1  -> /content/checkpoints-lightgcn/best.pt
Epoch 002/40 | loss=0.6700 | 527.4s
  [val] HR@1=0.0347  HR@5=0.1209  HR@10=0.1766  HR@20=0.2356  HR@50=0.3186
          ND@1=0.0347  ND@5=0.0492  ND@10=0.0609  ND@20=0.0725  ND@50=0.0875
  No improvement (1/5) | best=0.0784 @ epoch 1
Epoch 003/40 | loss=0.5301 | 526.3s
  [val] HR@1=0.0361  HR@5=0.1227  HR@10=0.1770  HR@20=0.2358  HR@50=0.3201
          ND@1=0.0361  ND@5=0.0506  ND@10=0.0620  ND@20=0.0736  ND@50=0.0886
  No improvement (2/5) | best=0.0784 @ epoch 1
Epoch 004/40 | loss=0.4433 | 526.3s
  [val] HR@1=0.0351  HR@5=0.1266  HR@10=0.1828  HR@20=0.2400  HR@50=0.3237
          

## 10. Test Set Evaluation với Best Checkpoint

In [32]:
import torch, os

ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded: epoch={ckpt["epoch"]} | val {primary}={ckpt[primary]:.4f}')

m_test = evaluate(model, TEST_GT, TRAIN_MASK,
                  ks=CFG['evaluation']['ks'],
                  user_batch=CFG['training']['eval_batch_size'])

print('\n' + '='*48)
print('TEST SET RESULTS')
print('='*48)
print(f'{"k":>6} | {"HR@k":>10} | {"NDCG@k":>10}')
print('-'*32)
for k in CFG['evaluation']['ks']:
    print(f'{k:>6} | {m_test[f"HR@{k}"]:>10.4f} | {m_test[f"NDCG@{k}"]:>10.4f}')
print('='*48)

Loaded: epoch=1 | val NDCG@20=0.0784

TEST SET RESULTS
     k |       HR@k |     NDCG@k
--------------------------------
     1 |     0.0150 |     0.0150
     5 |     0.0758 |     0.0314
    10 |     0.1230 |     0.0425
    20 |     0.1747 |     0.0532
    50 |     0.2478 |     0.0665


## 11. Summary

In [33]:
print('='*65)
print('EXPERIMENT SUMMARY')
print('='*65)
print(f'Model              : LightGCN')
print(f'Dataset            : BPATMP (REES46)')
print(f'Users / Items      : {N_USERS:,} / {N_ITEMS:,}')
print(f'Training data:')
print(f'  view (subsampled): {len(view_src):,}')
print(f'  cart (full)      : {len(cart_src):,}')
print(f'  purchase (full)  : {len(pur_src):,}')
print(f'  combined (dedup) : {len(train_users):,}')
print(f'Embed dim          : {CFG["model"]["embed_dim"]}')
print(f'GCN layers         : {CFG["model"]["n_layers"]}')
print(f'Best epoch         : {best_epoch}')
print(f'Val {primary:12s}  : {best_metric:.4f}')
print('-'*65)
print('Test metrics:')
for k in CFG['evaluation']['ks']:
    print(f'  HR@{k:<3} = {m_test[f"HR@{k}"]:.4f}   NDCG@{k:<3} = {m_test[f"NDCG@{k}"]:.4f}')
print('='*65)
print(f'Best checkpoint : {best_ckpt_path}')
print(f'Last checkpoint : {last_ckpt_path}')

EXPERIMENT SUMMARY
Model              : LightGCN
Dataset            : BPATMP (REES46)
Users / Items      : 203,063 / 29,892
Training data:
  view (subsampled): 3,000,000
  cart (full)      : 3,868,050
  purchase (full)  : 2,436,760
  combined (dedup) : 3,306,742
Embed dim          : 64
GCN layers         : 3
Best epoch         : 1
Val NDCG@20       : 0.0784
-----------------------------------------------------------------
Test metrics:
  HR@1   = 0.0150   NDCG@1   = 0.0150
  HR@5   = 0.0758   NDCG@5   = 0.0314
  HR@10  = 0.1230   NDCG@10  = 0.0425
  HR@20  = 0.1747   NDCG@20  = 0.0532
  HR@50  = 0.2478   NDCG@50  = 0.0665
Best checkpoint : /content/checkpoints-lightgcn/best.pt
Last checkpoint : /content/checkpoints-lightgcn/last.pt
